# 考试万能模板（离线 Colab / 本地 Jupyter）

考前把本文件下载到本地。开考先复制结构，再改列名。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

DATA = Path("../data")  # 考试时改成实际路径
compounds = pd.read_csv(DATA / "compounds.csv")
cell_lines = pd.read_csv(DATA / "cell_lines.csv")
sensitivity = pd.read_csv(DATA / "sensitivity.csv")

df = (
    sensitivity.dropna(subset=["IC50"])
    .drop_duplicates()
    .merge(compounds, on="drug_id", how="inner")
    .merge(cell_lines, on="cell_id", how="inner")
    .dropna(subset=["smiles"])
)
print(df.shape)
df.head()

In [ ]:
def smiles_to_fp(smiles, n_bits=32):
    vec = np.zeros(n_bits)
    tokens = list("CNOSF=#()[]123cno") + ["Cl", "Br"]
    for i, tok in enumerate(tokens):
        vec[i % n_bits] += str(smiles).count(tok)
    vec[-1] = len(str(smiles))
    return vec

X = np.vstack([smiles_to_fp(s) for s in df["smiles"]])
y = df["IC50"].values
X.shape, y.shape

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmses, r2s = [], []

for train_idx, test_idx in kf.split(X):
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X[train_idx], y[train_idx])
    y_pred = model.predict(X[test_idx])
    rmses.append(np.sqrt(mean_squared_error(y[test_idx], y_pred)))
    r2s.append(r2_score(y[test_idx], y_pred))

print("mean RMSE:", float(np.mean(rmses)))
print("mean R2:", float(np.mean(r2s)))